## **Maestría en Inteligencia Artificial Aplicada**
### **Curso: Inteligencia Artificial y Aprendizaje Automático**
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad de la Semana:**

### **Descomposición en Valores Singulares (SVD) y Sistemas de Recomendación**


**Esta Actividad deberá resolverse de manera individual.**

**Nombre y matrícula:**

* Mónica María Ramírez Mejía - A01797493   



# **Introducción**

In [108]:
# Agrega aquí todas las librerías y paquetes adicionales que requieras.

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity



* ### **Liga de datos de la UCI: "Restaurant & consumer data".**

https://archive.ics.uci.edu/dataset/232/restaurant+consumer+data

* ### **Del archivo comprimido "RCdata.zip" necesitamos solamente "rating_final.csv" y "geoplaces2.csv".**

* ### **NOTA: Al igual que con las variables numéricas, la información de variables con información de texto (strings) tiene sus propias técnicas de limpieza y transformación que podrás estudiar en cursos posteriores (en particular en el curso optativo de Procesamiento de Lenguaje Natural). No es el objetivo de este curso adentrarnos en dichas técnicas, pero sí requerimos hacer unos ajustes de limpieza para poder llevar a cabo la presente actividad. En particular, en la pregunta 2 te indicaré directamente cuál es el ajuste que hay que realizar para que el algoritmo de factorización SVD pueda funcionar más adelante.**

In [109]:
# Cargamos los archivos de la página de la UCI para construir
# nuestra matriz de utilidad:
from google.colab import drive
drive.mount('/content/drive')
import os
DIR = "/content/drive/MyDrive/Colab Notebooks/IA y Aprendizaje automatico/S8"
os.chdir(DIR)

data_rating = pd.read_csv("rating_final.csv", header='infer', sep=",")
data_geo = pd.read_csv("geoplaces2.csv", header='infer',  encoding='latin-1')

print(data_rating.shape, data_geo.shape)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(1161, 5) (130, 21)


# **Ejercicio - 1**

* ### **Explica cuál es el propósito del argumento "encoding" al cargar el segundo archivo. En particular, explica el error que te genera si omitimos dicho argumento y por qué el primer archivo no lo requiere.**

++++++++ Inicia la sección de agregar texto: +++++++++++

Este argumento se usa para indicar como están guardados los caracteres dentro del archivo "geoplaces2". En este caso, se debe especificar "encoding= latin-1" porque el archivo contiene direcciones y, si no se coloca, puede aparecer un error al intentar leer las que contengan tildes u otros caracteres especiales. En el caso del archivo "rating_final" no tiene valores con caracteres especiales, por lo que, no es necesario agregar el argumento.

++++++++ Termina la sección de agregar texto. +++++++++++

In [110]:
# Veamos la lista de los nombres de los restaurantes de nuestros datos.
# Todos ellos son restaurantes ubicados en alguna ciudad del país de México:

data_geo['name'].values

array(['Kiku Cuernavaca', 'puesto de tacos', 'El Rincón de San Francisco',
       'little pizza Emilio Portes Gil', 'carnitas_mata',
       'Restaurant los Compadres', 'Taqueria EL amigo ', 'shi ro ie',
       'Pollo_Frito_Buenos_Aires', 'la Estrella de Dimas',
       'Restaurante 75', 'Abondance Restaurante Bar',
       'El angel Restaurante', 'Restaurante Pueblo Bonito',
       'Mcdonalds Parque Tangamanga', 'Tortas y hamburguesas el gordo',
       'Sirlone', 'rockabilly ', 'Unicols Pizza', 'TACOS EL GUERO',
       'Restaurant El Muladar de Calzada', 'La Posada del Virrey',
       'Restaurant and Bar and Clothesline Carlos N Charlies', 'KFC',
       'Giovannis', 'Restaurant Oriental Express', 'Mariscos Tia Licha',
       'cafe ambar', 'Restaurante la Gran Via', 'don burguers',
       'Restaurante y Pescaderia Tampico', 'Rincon del Bife',
       'La Fontana Pizza Restaurante and Cafe',
       'Restaurante la Estrella de Dima', 'El Rincon de San Francisco',
       'Preambulo Wifi Zone 

#### **Para que el algoritmo SVD funcione, cada registro (renglón) debe estar asociado a un restaurante con un nombre diferente.**

### **Algunos comentarios acerca de estos nombres. Para obtener la información de los siguientes comentarios en realidad requerimos algunas técnicas de procesamiento de lenguaje natural, pero como te comentaba, por el momento te las proporciono directamente para no distraernos con dihcas técnicas de limpieza:**

*   ### **En la presente actividad generaremos un sistema de recomendación con base a la evaluación de varias características del servicio o tipo de comida de cada restaurante. Por el momento no tomaremos en cuenta la ubicación para generar nuestro sistema de recomendación, pero sí necesitamos aclarar ciertos puntos que menciono a continuación.**

* ### **La longitud y latitud es información que encuentras en el archivo geoplaces2.csv y son las coordenadas de la ubicación geográfica de cada restaurante. Se puede verificar que todas las coordenadas son diferentes, es decir, que todos los registros (renglones) contienen información de restaurantes ubicados en diferentes lugares de México.**

*   ### **Algunos restaurantes pertenezcan a una misma franquicia, como por ejemplo, Vips.**

*   ### **Para los fines de este ejercicio, cada registro de la base de datos lo consideraremos como un negocio diferente, independientemente de que pertenezcan a una misma franquicia. Así, en particular en el caso de Vips, existen tres restaurantes en esta lista capturados con los nombres: VIPS, Vips y vips. Por sus coordenadas de latitud y longitud sabemos que están ubicados en San Luis Potosí, Cuernavaca y Ciudad Victoria, tres ciudades diferentes de México. Como estos tres nombres están escritos de manera diferente, al momento de procesarlos con el algoritmo de SVD se estarán considerando como tres restaurantes diferentes. Por el momento es lo que queremos y por lo tanto no haremos ajustes en dichos nombres.**

*   ### **En el archivo existen más restaurantes pertenecientes a otras franquicias que han sido capturados también de manera diferente y nuevamente, así los dejaremos. Sin embargo, en el caso de 'Gorditas Dona Tota', dos de estos restaurantes están ubicados en Cd.Victoria y están escritos exactamente de la misma forma. Si los dejamos de esta manera, nuestro algoritmo SVD no funcionará, por ello, debes cambiar el nombre de uno de ellos como se te indica a continuación.**  

# **Ejercicio - 2**

* ### **Encuentra los índices de los dos restaurantes registrados exactamente con el mismo nombre de 'Gorditas Dona Tota' y cambia el nombre del que tiene el mayor índice en data_geo[name], al de  'Gorditas Dona Tota 2'.**

In [111]:
# Ejercicio-2

# ************* Inlcuye aquí tu código:*****************************


# Incluyamos en la siguiente variable la lista de todos los índices
# de restaurantes con exactamente el mismo nombre de 'Gorditas Dona Tota':

indices_Dona_Tota = data_geo[data_geo['name'] == 'Gorditas Dona Tota'].index
data_geo.loc[max(indices_Dona_Tota), 'name'] = 'Gorditas Dona Tota 2'


# *********** Aquí termina la sección de agregar código *************

print('Desplegando los nombres de los dos restaurantes con el ajuste:\n', data_geo.loc[indices_Dona_Tota, 'name'])


Desplegando los nombres de los dos restaurantes con el ajuste:
 89       Gorditas Dona Tota
118    Gorditas Dona Tota 2
Name: name, dtype: object


In [112]:
# Al momento tenemos los siguientes DataFrames. Del primer archivo
# tenemos la evaluación general, la de la comida y la del servicio:

data_rating.head(3)

,userID,placeID,rating,food_rating,service_rating
0,U1077,135085,2,2,2
1,U1077,135038,2,2,1
2,U1077,132825,2,2,2


In [113]:
# Y del segundo archivo obtenemos información diversa de cada restaurante:

data_geo.head(2).T

,0,1
placeID,134999,132825
latitude,18.915421,22.147392
longitude,-99.184871,-100.983092
the_geom_meter,0101000020957F000088568DE356715AC138C0A525FC46...,0101000020957F00001AD016568C4858C1243261274BA5...
name,Kiku Cuernavaca,puesto de tacos
address,Revolucion,esquina santos degollado y leon guzman
city,Cuernavaca,s.l.p.
state,Morelos,s.l.p.
country,Mexico,mexico
fax,?,?


# **Ejercicio - 3**

* ### **De cada uno de estos archivos selecciona y combina las variables adecuadas para obtener un nuevo DataFrame con 4 columnas: el ID de usuario (userID), el ID del restaurante (placeID), la calificación general (rating) y el nombre del restaurante (name). A este nuevo DataFrame llamarlo df_combinado.**

In [114]:
# Ejercicio 3:

# ************* Inlcuye aquí tu código:*****************************

df_combinado = pd.merge(data_rating, data_geo, on='placeID')
df_combinado = df_combinado[['userID','placeID','rating','name']]

# *********** Aquí termina la sección de agregar código *************


# Despleguemos la dimensión y los primeros renglones de este DataFrame:

print(df_combinado.shape)
df_combinado.head()

(1161, 4)


,userID,placeID,rating,name
0,U1077,135085,2,Tortas Locas Hipocampo
1,U1077,135038,2,Restaurant la Chalita
2,U1077,132825,2,puesto de tacos
3,U1077,135060,1,Restaurante Marisco Sam
4,U1068,135104,1,vips


# **Ejercicio - 4**

In [115]:
# Ejercicio 4:

#    Define la matriz de utilidad cuyos renglones sean los nombres de los
#    restaurantes, las columnas los IDs de los usuarios y las entradas la
#    evaluación general (rating). La llamaremos "UtMx_rating".


# ************* Inlcuye aquí tu código:*****************************

UtMx_rating = df_combinado.pivot_table(index='name', columns='userID', values='rating')
UtMx_rating = UtMx_rating.fillna(0)

# *********** Aquí termina la sección de agregar código *************


print('Dimensión de la matriz de Utilidad:')
print('(restaurantes, usuarios) =', (UtMx_rating.shape))
UtMx_rating.head()

Dimensión de la matriz de Utilidad:
(restaurantes, usuarios) = (130, 138)


userID,U1001,U1002,U1003,U1004,U1005,U1006,U1007,U1008,U1009,U1010,...,U1129,U1130,U1131,U1132,U1133,U1134,U1135,U1136,U1137,U1138
name,,,,,,,,,,,,,,,,,,,,,
Abondance Restaurante Bar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Arrachela Grill,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cabana Huasteca,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0
Cafe Chaires,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cafeteria cenidet,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### **Similaridades con la matriz obtenida de la Factorización SVD Truncada:**

*   ### **Aplicaremo la Factorización SVD con los 66 primeros valores singulares a la matriz de utilidad UtMx_rating.**

*   ### **El valor de 66 se seleccionó porque esta cantidad de valores singulares es suficiente para describir al menos el 95% de la varianza de la matriz de utilidad (información que mostraremos a continuación).**

*   ### **Recordemos que la factorización SVD de una matriz $A$ tiene la forma:** $A_{m\times n} = U_{m\times m}\Sigma_{m\times n}V_{n\times n}^T$

*   ### **Usaremos la función coseno como medida de similaridad**

*   ### **Usaremos el sistema de recomendación construído con SVD para buscar restaurantes similares al restaurante llamado  "tacos de barbacoa enfrente del Tec"**.

In [116]:
num_componentes=66  # componentes más significativas

# Obtengamos la matriz de factores latentes de los restaurantes:
SVD_rating = TruncatedSVD(n_components=num_componentes)  # inicializamos
matriz_latente_restaurantes = SVD_rating.fit_transform(UtMx_rating) # SVD truncada-ndarray

print('Dimensión Vectores latentes restaurantes:', matriz_latente_restaurantes.shape)
print('Dimensión Traspuesta Vectores latentes usuarios:', SVD_rating.components_.shape)

# Calculemos la variabilidad acumulada de las componentes utilizadas:
print('\nVariabilidad acumulada de las componentes utilizadas: %.3f' % SVD_rating.explained_variance_ratio_[0:num_componentes].sum())

# matriz de similaridad
sim_matrix = cosine_similarity(matriz_latente_restaurantes)

# Restaurante de referencia:
restaurante_de_referencia = "tacos de barbacoa enfrente del Tec"
nombres_rest = UtMx_rating.T.columns  # lista de los nombres de los restaurantes.
idx_rest = list(nombres_rest).index(restaurante_de_referencia) # índice del restaurante de referencia.
sim_vector = sim_matrix[idx_rest] # Vector de similaridad del restaurante de referencia contra todos.

# Buscando las similaridades positivas:
idx = (sim_vector>0)
mejores_sim_rate = list()
for i in range(len(nombres_rest[idx])):
  mejores_sim_rate.append((sim_vector[idx][i], nombres_rest[idx][i]))

print('\nTotal de similaridades positivas encontradas:', len(mejores_sim_rate))

# Las ordenamos de mayor a menor relevancia:
mejores_sim_rate_ordenadas = sorted(mejores_sim_rate, key=lambda x:x[0], reverse=True)

print('\nMejores recomendaciones con base al restaurante: \"%s\":' % restaurante_de_referencia)
mejores_sim_rate_ordenadas[1:11]   # omitimos el primero que es el mismo restaurante de referencia


Dimensión Vectores latentes restaurantes: (130, 66)
Dimensión Traspuesta Vectores latentes usuarios: (66, 138)

Variabilidad acumulada de las componentes utilizadas: 0.952

Total de similaridades positivas encontradas: 82

Mejores recomendaciones con base al restaurante: "tacos de barbacoa enfrente del Tec":


[(np.float64(0.9372899041765831), 'vips'),
 (np.float64(0.9343035937242763), 'little pizza Emilio Portes Gil'),
 (np.float64(0.9267746795382874), 'tacos abi'),
 (np.float64(0.5297850733511527), 'Carreton de Flautas y Migadas'),
 (np.float64(0.4981661248179026), 'puesto de gorditas'),
 (np.float64(0.47750800397327225), 'Taqueria EL amigo '),
 (np.float64(0.3646329193176536), 'carnitas_mata'),
 (np.float64(0.26240182632878856), 'Little Cesarz'),
 (np.float64(0.2335213557590398), 'Gorditas Dona Tota 2'),
 (np.float64(0.184251788617734), 'carnitas mata calle Emilio Portes Gil')]

# **Ejercicio - 5**

### **Sistema de Recomendación Híbrido**

* ### **Construyamos ahora un sistema de recomendación híbrido, el cual consistirá en conjuntar "matriz_latente_restaurantes" (que es un ndarray) obtenida previamente, con la matriz de factores adicionales (que también será un ndarray) formada por las columnas One-Hot-Encoder de los factores 'dress_code', 'accessibility', 'price' y 'franchise' del DataFrame data_geo.**

* ### **Los renglones de la matriz híbrida siguen representando a cada restaurante diferente, pero ahora con la información conjunta de los vectores latentes de la factorización SVD, junto con los vectores OneHotEncoding de los nuevos factores que incluídos. Esta combinación de información es una de las técnicas clásicas para contruir sistemas de recomendación más robustos.**



In [117]:
# Ejercicio 5:

# Define el DataFrame "df_fact_adicionales" formada por los factores 'dress_code',
# 'accessibility', 'price' y 'franchise'.
# Después genera la matriz (ndarray), "matriz_factores_adicionales_ohe", que se obtiene
# al transformar las columnas (variables categóricas nominales) de "df_fact_adicionales"
# mediante OneHotEncoding.
# Finalmente conjuntar horizontalmente la matriz "matriz_latente_restaurantes"
# con "matriz_factores_adicionales_ohe" para obtener la "matriz_hibrida" (ndarray).
# Toma en cuenta que "matriz_hibrida" estará formada por las columnas de
# "matriz_latente_restaurantes" seguida por las columnas de "mat_fact_adicionales_ohe".


# ************* Inlcuye aquí tu código:*****************************

df_fact_adicionales = data_geo[['dress_code','accessibility','price','franchise']]
matriz_factores_adicionales_ohe = pd.get_dummies(df_fact_adicionales).values
matriz_hibrida = np.hstack((matriz_latente_restaurantes, matriz_factores_adicionales_ohe))

# *********** Aquí termina la sección de agregar código *************


print('Dimensión de la matriz híbrida:')
matriz_hibrida.shape

Dimensión de la matriz híbrida:


(130, 77)

In [118]:
# Recordemos que "restaurante_de_referencia" es "tacos de barbacoa enfrente del Tec".
# Obtenengamos el índice de este restaurante:

idx = data_geo[data_geo['name'] == restaurante_de_referencia].index[0]

print('Restaurante de referencia:', restaurante_de_referencia)
print('Índice del restaurante de referencia:', idx)

Restaurante de referencia: tacos de barbacoa enfrente del Tec
Índice del restaurante de referencia: 111


# **Ejercicio - 6:**

* ### **Ahora que tienes la matriz híbrida, con dicha matriz obtener las similitudes (con respecto a la función coseno) del restaurante de referencia "tacos de barbacoa enfrente del Tec" con relación a todos los restaurantes. Deberás desplegar los primeros 10 restaurantes de mayor similitud (sin incluir al restaurante de referencia mismo), ordenados de mayor a menor similitud, y también desplegando el valor de similitud mismo.**

In [119]:
# Ejercicio 6

# ************* Inlcuye aquí tu código:*****************************

matriz_similitud_hibrida = cosine_similarity(matriz_hibrida)
vector_similitud = matriz_similitud_hibrida[idx]
resultados = []

for i in range(len(vector_similitud)):
    resultados.append((vector_similitud[i], data_geo['name'].iloc[i]))
resultados_ordenados = sorted(resultados, reverse=True)

for r in resultados_ordenados[1:11]:
    print(r)

# *********** Aquí termina la sección de agregar código *************


(np.float64(0.46532796680946303), 'Arrachela Grill')
(np.float64(0.45176693880217067), 'palomo tec')
(np.float64(0.4434308228468745), 'Pizzeria Julios')
(np.float64(0.4253576177371674), 'TACOS CORRECAMINOS')
(np.float64(0.4176021595261118), 'Gorditas Dona Tota 2')
(np.float64(0.40532307798672235), 'tortas hawai')
(np.float64(0.3818014041794942), 'Tortas y hamburguesas el gordo')
(np.float64(0.38103866537597875), 'Vips')
(np.float64(0.3734739034229226), 'tacos abi')
(np.float64(0.34641043288515316), 'Michiko Restaurant Japones')


# **Ejercicio - 7**

### **Incluye tus comentarios y conclusiones de la actividad.**

++++++++ Inicia la sección de agregar texto: +++++++++++

Gracias a este ejercicio pude comprender como se aplica la técnica SVD. Con esta es posible analizar las preferencias de los usuarios/consumidores a partir de sus calificaciones y encontrar similitudes entre el servicio o producto que se está analizando, en este caso restaurantes. En general, la actividad me permitió entender como se puede analizar datos para desarrollar sistemas de recomendación con base en las valoraciones proporcionadas por los usuarios.

++++++++ Termina la sección de agregar texto. +++++++++++

# **++++ Fin de la Actividad de la semana: Sistema de Recomendación Híbrido +++**